# Set orientation

This notebook measures how your camera's picture is turned — and possibly
mirrored — relative to the way the stage moves, and stores the answer so
that every image the driver saves comes out aligned with the stage axes.

Why this matters: when automation finds a cell in an image and asks the
stage to move there, it converts pixel positions into stage moves. If the
camera is rotated or mirrored relative to the stage, that conversion walks
the wrong way — target acquisition would chase every cell in the wrong
direction. Measuring the turn once fixes the conversion for every later
session on this microscope.

Run this after setting the stage limits. Park on any spot with visible
texture (cells, debris, a felt-pen dot on a slide — anything the camera can
latch onto) and focus. The measurement takes three quick pictures and moves
the stage by a few tens of micrometres.


## Configure

Edit the values and run this cell — nothing is acquired yet.

- `session_id` names this measurement's folder; put today's date in it.
- `job_name` is the Navigator Expert job to image with. Select it once in
  LAS X and keep it selected for the whole notebook.
- `reference_objective` is recorded in the report: write the name of the
  objective you have in. (The turn is a property of the camera and stage,
  not of the lens, so any objective works.)
- `stage_move_um` is how far the stage steps during the measurement. A
  bigger step is easier to detect, but the spot you parked on must stay
  inside the picture — about a tenth of the field of view works well.


In [ ]:
import _bootstrap
from IPython.display import Image, display
from navigator_expert.config.machine import MACHINE
from navigator_expert.orientation import measure as wf

session = wf.start_session(
    session_id="2026-05-22_scope_orientation",
    job_name="Overview",
    reference_objective="10x",
    stage_move_um=40.0,
    sessions_root=MACHINE.work_root("orientation"),
)
session


## Measure

This cell takes three pictures: one at the starting spot ("home"), one
after a small stage step in +X, and one after a step in +Y. From the way
the picture content appears to move, the driver works out the camera's
turn — one of eight possibilities: four rotations (0°, 90°, 180°, 270°),
each either plain or mirrored.

The gallery below shows all eight candidates side by side. In each panel
the two stepped pictures are shifted back by where *that* candidate says
the content should have moved, and overlaid on the home picture (magenta =
home, green = stepped). Only the true orientation lines the pictures up —
that panel turns white/grey where the images overlap and is drawn with a
highlighted border. Wrong candidates stay visibly magenta/green.

If the measurement is rejected, the usual causes are too little texture at
the chosen spot or a stage step too small to measure reliably. Move to a
spot with more detail, or increase `stage_move_um`, and run this cell
again.


In [ ]:
session = wf.measure(session)
diagnostic = session.paths.reports_dir / wf.DIAGNOSTIC_NAME
if diagnostic.exists():
    display(Image(filename=str(diagnostic)))
session.orientation if session.config_written else session.failure_reason


## Save and adopt

Adopting makes this measurement the microscope's active orientation: the
result is published into the machine's configuration folder together with
a copy of this notebook, so the folder itself documents how the value was
measured.

Run the first cell below, save this notebook with `Ctrl+S`, then run the
final cell. The save matters: the archived copy must contain the gallery
you just looked at. Browser-based Jupyter may save automatically, but VS
Code needs the manual save. Adoption stays blocked until the saved
notebook includes this measurement.


In [ ]:
previous_notebook_mtime_ns = _bootstrap.request_notebook_save()


In [ ]:
saved_notebook = _bootstrap.wait_for_notebook_save(previous_notebook_mtime_ns)
wf.adopt_orientation(session, notebook_paths=[saved_notebook])
